# 05. Gating Extracted Invoice Text with Azure AI Content Safety

This notebook combines two services from earlier in the section: it re-runs the **Azure AI Document Intelligence** `prebuilt-layout` extraction from `01_document_intelligence.ipynb` against a sample CloudXeus invoice, then feeds the extracted text through **Azure AI Content Safety**'s `analyze_text` moderation call before deciding whether that text is safe to hand to an LLM agent. This is a standard defensive pattern for document-processing pipelines: **never pass externally-sourced, untrusted document text (a PDF someone uploaded) straight into an LLM prompt without a moderation gate first** — the same principle underlies why `04_invoke_agent.ipynb`'s agent only receives data that's already been through a structured extraction step.

**Difficulty: Beginner/Intermediate**

## Prerequisites

**Python packages**
```bash
pip3 install azure-ai-documentintelligence azure-ai-contentsafety python-dotenv
```
`azure-ai-contentsafety` is **not** currently in the repo root `requirements.txt` — install it explicitly alongside `azure-ai-documentintelligence` (also missing from root `requirements.txt`; see `01_document_intelligence.ipynb`).

**Azure resources**
- An **Azure AI Document Intelligence** resource (same as notebook 1).
- A separate **Azure AI Content Safety** resource, with its own endpoint and key.

**Environment variables** (add to the root `.env`):

```
AZURE_DOCUMENTINTELLIGENCE_ENDPOINT=https://<your-resource>.cognitiveservices.azure.com/
AZURE_DOCUMENTINTELLIGENCE_KEY=<your-key>
AZURE_CONTENTSAFETY_ENDPOINT=https://<your-safety-resource>.cognitiveservices.azure.com/
AZURE_CONTENTSAFETY_KEY=<your-key>
DOCUMENT_URL=https://<your-storage-account>.blob.core.windows.net/.../CloudXeus_Invoice_INV-CX-2026-1001.pdf
```

The original script analyzes `CloudXeus_Invoice_INV-CX-2026-1001.pdf`; a local copy also lives in `../cloudxeus_invoices/`.

## What You'll Learn

- Chaining two Azure AI services: extraction (Document Intelligence) → moderation (Content Safety)
- `AnalyzeTextOptions` and `ContentSafetyClient.analyze_text`
- The shape of a Content Safety result: one `category_result` (category name + severity) per harm category
- Azure AI Content Safety's severity scale, and how to use it to build a pass/block gate
- A real, easy-to-miss Python indentation bug — and why careful review of loop bodies matters even in short scripts

### Imports and environment-driven configuration

This reuses notebook 1's Document Intelligence setup, plus a second client/config pair for Content Safety.

In [ ]:
import os

from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.core.credentials import AzureKeyCredential
from azure.ai.contentsafety import ContentSafetyClient
from azure.ai.contentsafety.models import AnalyzeTextOptions
from dotenv import load_dotenv

load_dotenv()

CONTENT_SAFETY_ENDPOINT = os.getenv("AZURE_CONTENTSAFETY_ENDPOINT")
CONTENT_SAFETY_KEY = os.getenv("AZURE_CONTENTSAFETY_KEY")

endpoint = os.getenv("AZURE_DOCUMENTINTELLIGENCE_ENDPOINT")
key = os.getenv("AZURE_DOCUMENTINTELLIGENCE_KEY")

# Sample CloudXeus invoice PDF. Local copy: ../cloudxeus_invoices/CloudXeus_Invoice_INV-CX-2026-1001.pdf
document_url = os.getenv(
    "DOCUMENT_URL",
    "https://stcxai103capdeus01.blob.core.windows.net/course-products/student-invoices/CloudXeus_Invoice_INV-CX-2026-1001.pdf",
)


### Extracting the invoice text with Document Intelligence

Identical pattern to notebook 1: `begin_analyze_document` with `prebuilt-layout`, blocking on `poller.result()`, then taking `document_result.content` as the plain extracted text/markdown we're about to moderate.

In [ ]:
client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key),
    api_version="2024-11-30",
)

poller = client.begin_analyze_document(
    model_id="prebuilt-layout",
    body=AnalyzeDocumentRequest(url_source=document_url),
    output_content_format="markdown",
)

document_result = poller.result()

extracted_text = document_result.content


### Moderating the extracted text with Content Safety

`ContentSafetyClient.analyze_text` takes an `AnalyzeTextOptions` wrapping the text to check, and returns a result with one entry in `categories_analysis` per harm category. Each entry has a `category` (name) and a `severity` (an integer on Content Safety's severity scale).

💡 **Exam tip (AI-102):** Azure AI Content Safety's `analyze_text` evaluates four harm categories — **Hate**, **SelfHarm**, **Sexual**, and **Violence** — and by default reports severity on a four-level scale (0 = Safe, 2 = Low, 4 = Medium, 6 = High) for each category independently; a more granular 0–7 scale is also available via the API's `output_type` parameter. Building a "gate" that blocks content above a chosen severity threshold (as this script does, threshold `>= 4`) before that content reaches an LLM prompt is exactly the kind of responsible-AI content-moderation pattern the exam expects you to recognize.

⚠️ **A real bug worth noticing:** in the loop below, look closely at the indentation. `if category_result.severity >= 4:` is **not** indented inside the `for` loop — it runs once, *after* the loop finishes, using whatever `category_result` the loop happened to leave behind (its last iteration). That means this script only ever checks the **last** category returned, not all four. We're preserving this exactly as it appears in the original source script (faithful conversion, no functional changes) — but it's a genuinely useful bug to spot, and fixing it is this notebook's first **Try It Yourself** exercise.

🔄 **Alternatives:** Beyond category/severity moderation, Content Safety also offers **Prompt Shields**, which specifically detects prompt-injection and jailbreak attempts embedded in untrusted text — highly relevant here, since this pipeline eventually feeds extracted PDF text into an LLM agent, and a malicious PDF could in principle try to inject instructions rather than (or in addition to) containing merely "unsafe" content.

In [ ]:
content_safety_client = ContentSafetyClient(
    endpoint=CONTENT_SAFETY_ENDPOINT,
    credential=AzureKeyCredential(CONTENT_SAFETY_KEY),
)

text_request = AnalyzeTextOptions(
    text=extracted_text
)

moderation_result = content_safety_client.analyze_text(text_request)

print("\nContent Safety result:")
should_block = False

for category_result in moderation_result.categories_analysis:
    print(category_result.category, category_result.severity)
if category_result.severity >= 4:
        should_block = True


if should_block:
    print("\nBlocked: unsafe content was detected in the product PDF.")
else:
    print("\nAllowed: the product PDF text can be passed to the agent.")


## Summary

We re-extracted a CloudXeus invoice's text with Document Intelligence, then ran that text through Azure AI Content Safety's `analyze_text`, printing each harm category's severity score and computing a `should_block` flag from a severity-4-or-higher threshold. The printed "Blocked" / "Allowed" message is the pipeline's answer to "is this externally-sourced document text safe to hand to an LLM agent?" — the same kind of gate you'd want in front of `04_invoke_agent.ipynb`'s prompt-building step in a production pipeline, even though these two notebooks don't currently call each other directly.

## Try It Yourself

1. **Easy (bug fix):** Fix the indentation bug — move `if category_result.severity >= 4: should_block = True` fully inside the `for` loop so every category is actually checked, not just the last one. Re-run and confirm `should_block` can now be set by *any* category, not only the final one printed.
2. **Intermediate:** Make the severity threshold configurable via an environment variable (e.g. `CONTENT_SAFETY_BLOCK_THRESHOLD`, default `4`), and print which specific category (or categories, after the bug fix) triggered a block.
3. **Advanced:** Add a Content Safety **Prompt Shields** check (a separate API call) alongside the category/severity check, so the gate catches both "unsafe content" and "attempted prompt injection" before the text is ever passed into `04_invoke_agent.ipynb`'s prompt.